# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
# Print name and description from the metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Citation: {getattr(metadata, 'cite_as', getattr(metadata, 'citeAs', None))}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All references use the `@id` field for consistency per FAIR principle and the dataset's Croissant schema.

In [ ]:
# List all available record sets in the dataset with their @id and name
record_sets = dataset.metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}, Name: {getattr(rs, 'name', rs.id)}")
    if hasattr(rs, 'description'):
        print(f"    Description: {rs.description}")
    fields = getattr(rs, 'fields', [])
    print("    Fields:")
    for f in fields:
        print(f"        - Field @id: {f.id}, Name: {getattr(f, 'name', f.id)}, Data type: {getattr(f, 'data_type', None)}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

_Below, we first select the main protein abundance record set (replace `record_set_id` by the selected `@id` as shown in previous code output). We collect all record sets for illustration, but will explore one main table for EDA._

In [ ]:
# Extract data from all record sets
# (pick the main record set for downstream EDA)
dataframes = {}
from pprint import pprint

for rs in dataset.metadata.record_sets:
    print(f"Loading RecordSet @id: {rs.id} ...")
    try:
        records = list(dataset.records(record_set=rs.id))
        df = pd.DataFrame(records)
        dataframes[rs.id] = df
        print(f"    -> Loaded {len(df)} records with columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"    -> Failed to load: {e}")

# For this dataset, main table assumed to be @id matching abundance matrix or similar.
# Based on the schema, find the main RecordSet. (Choose the one with the most columns/records)
main_record_set_id = max(dataframes, key=lambda k: len(dataframes[k].columns))

print(f"\nMain RecordSet for EDA (by max columns): {main_record_set_id}\nColumns:")
pprint(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

_Fields for EDA are addressed by their `@id`, as per the Croissant schema. Adjust field names to match the @id of numeric and categorical fields in your table. Here we attempt to select some likely columns (e.g., `cr:peptide_count`, `cr:abundance`, and a sample grouping field such as `cr:sample_id`)._


In [ ]:
# Example field ids, adjust as appropriate for the dataset
main_df = dataframes[main_record_set_id]

# Pick a numeric field for filtering/normalizing (replace with actual @id from previous cell's columns)
potential_numeric_fields = [c for c in main_df.columns if ('abundance' in c.lower() or 'count' in c.lower() or 'intensity' in c.lower() or 'coverage' in c.lower() or 'mw' in c.lower())]
print(f"Numeric-like fields: {potential_numeric_fields}")
numeric_field_id = potential_numeric_fields[0] if len(potential_numeric_fields) else main_df.columns[0] # fallback

# Pick a group/categorical field (e.g., sample or description id)
potential_group_fields = [c for c in main_df.columns if c not in [numeric_field_id] and ('sample' in c.lower() or 'id' in c.lower() or 'group' in c.lower() or 'condition' in c.lower())]
group_field_id = potential_group_fields[0] if len(potential_group_fields) else main_df.columns[0] # fallback

print(f"Selected numeric field for analysis: {numeric_field_id}")
print(f"Selected group field for grouping: {group_field_id}")

# Filter for records with field value above a threshold
threshold = main_df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(main_df[numeric_field_id]) else 0
try:
    filtered_df = main_df[pd.to_numeric(main_df[numeric_field_id], errors='coerce') > threshold].copy()
except Exception:
    filtered_df = main_df.copy()

print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize numeric field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[f"{numeric_field_id}_normalized"] - filtered_df[f"{numeric_field_id}_normalized"].mean()) / filtered_df[f"{numeric_field_id}_normalized"].std()

print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by another field
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id, dropna=False)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize the filtered and normalized numeric field distribution, and the group means where possible.

_This example creates a histogram for the main numeric field, and a bar plot for group means._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field (normalized)
plt.figure(figsize=(7, 4))
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), kde=True)
plt.title(f"Distribution of normalized {numeric_field_id}")
plt.xlabel(f"{numeric_field_id}_normalized")
plt.ylabel("Count")
plt.show()

# (Optional) Bar plot of group means if grouping field exists
if 'grouped_df' in locals():
    plt.figure(figsize=(8, 4))
    sns.barplot(data=grouped_df.sort_values(numeric_field_id, ascending=False), x=group_field_id, y=numeric_field_id)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we used the FAIR² Croissant-metadata-powered dataset to:

- Load metadata and enumerate record sets using their `@id`.
- Load records into DataFrames using `mlcroissant` and column `@id`s.
- Explore and visualize protein abundance or other numeric fields (all fields referenced by their `@id`).
- Filter, normalize, and group records for basic data analysis.

This notebook can be extended to deeper analyses, leveraging the rich, machine-actionable structure provided by the Croissant schema and `mlcroissant` APIs.